# 10.3 Das Runge-Kutta-Verfahren

In Abschnitt 10.1 haben wir das Euler-Verfahren als Schleife implementiert.
In Abschnitt 10.2 haben wir gesehen, dass zu große Schrittweiten
unphysikalische Ergebnisse erzeugen, und dass selbst bei kleinen Schrittweiten
ein merklicher Fehler bleibt. Beide Probleme lassen sich mit
`scipy.integrate.solve_ivp` lösen. Die Funktion löst dieselbe DGL, wählt die
Schrittweite aber automatisch und nutzt intern das
**Runge-Kutta-Verfahren der Ordnung 4/5** (RK45): ein Verfahren, das pro Schritt
zwar mehr Rechenaufwand kostet als Euler, aber eine drastisch höhere Genauigkeit
erreicht.

## Lernziele

* [ ] Sie können `scipy.integrate.solve_ivp` mit dem Solver `RK45` aufrufen
  und die Rückgabefelder `sol.t`, `sol.y` und `sol.success` auslesen.
* [ ] Sie wissen, warum `f(t, y)` eine bestimmte Signatur haben muss, und
  können diese Funktion für eine skalare DGL korrekt schreiben.
* [ ] Sie können erklären, was **adaptive Schrittweite** bedeutet, und
  begründen, warum sie effizienter ist als eine feste Schrittweite.
* [ ] Sie können eine **nicht-autonome Differentialgleichung** (rechte Seite
  hängt explizit von $t$ ab) mit `solve_ivp` lösen.

## solve_ivp für das Abkühlproblem

*Wie viel einfacher kann man eine DGL lösen?* Wir nehmen dasselbe Kühlproblem
aus Abschnitt 10.1 und lösen es mit `solve_ivp`. Dazu importieren wir zunächst
die notwendigen Module und definieren die Abkühlung als Funktion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.style as style
from scipy.integrate import solve_ivp
style.use('seaborn-v0_8')

# --- Dieselben Parameter wie in Abschnitt 10.1 ---
T0_wert = 80.0   # Anfangstemperatur in °C
T_inf   = 20.0   # Umgebungstemperatur in °C
k       = 0.1    # Abkühlkonstante in 1/min

# --- Rechte Seite der DGL als Python-Funktion ---
# Pflichtformat: f(t, y)  ->  t ist Skalar, y ist Array
# Auch für eine skalare DGL: y = [T_aktuell], Rückgabe = [dT/dt]
# Wichtig: t steht an erster Stelle, auch wenn die DGL autonom ist (kein t auf der rechten Seite).
def f_abkuehlung(t, y):
    T_aktuell = y[0]                      # Temperatur aus dem Zustandsvektor
    dTdt = -k * (T_aktuell - T_inf)       # Newtonsche Abkühlungsrate
    return [dTdt]                         # Rückgabe als Liste mit einem Eintrag

`solve_ivp` erfordert einige Argumente. Wir übergeben nicht nur die
Differentialgleichung als Funktion `f_abkuehlung`, sondern auch das
Integrationsintervall `t_span`, die Anfangsbedingung `y0` und die Zeitpunkte
`t_eval`, zu denen die Lösung der Differentialgleichung ausgewertet werden soll.

In [ ]:
# --- solve_ivp Aufruf ---
# fun:    rechte Seite der DGL, Signatur f(t, y)
# t_span: Integrationsintervall als Tupel (Anfang, Ende)
# y0:     Anfangsbedingung als Liste; y0 = [T0] für eine skalare DGL
# t_eval: gewünschte Ausgabezeitpunkte (intern werden andere Punkte berechnet)
t_auswertung = np.linspace(0, 50, 501)
sol = solve_ivp(fun=f_abkuehlung, t_span=(0, 50), y0=[T0_wert], t_eval=t_auswertung)

Nachdem die Differentialgleichung gelöst wurde, können wir die Lösung über die
Attribute einsehen.

In [ ]:
# --- Rückgabe auslesen ---
# sol.t:    Ausgabezeitpunkte (= t_auswertung, da t_eval angegeben)
# sol.y:    Array der Form (Anzahl_Zustände, Anzahl_Zeitpunkte)
# sol.y[0]: erster Zustand (Temperatur) über die Zeit
# sol.success: True wenn die Integration erfolgreich war
print(f"Integration erfolgreich:  {sol.success}")
print(f"sol.y.shape:              {sol.y.shape}  "
      f"--> {sol.y.shape[0]} Zustand, {sol.y.shape[1]} Zeitpunkte")

Als nächstes analysieren wir die Genauigkeit der numerischen Lösung und
vergleichen dazu die Temperatur nach 10 min Abkühlung mit der exakten Lösung.

In [ ]:
# --- Genauigkeitsvergleich bei t = 10 min ---
# t_auswertung[100] = 10.0 min (linspace(0,50,501), Abstand = 0.1 min)
T_exakt_10  = T_inf + (T0_wert - T_inf) * np.exp(-k * 10.0)
T_ivp_10    = sol.y[0, 100]

# Euler mit dt = 5 min als Referenz (zwei Schritte bis t = 10 min)
T_euler_05  = T0_wert + 5.0 * (-k * (T0_wert - T_inf))   # t = 5 min
T_euler_10  = T_euler_05 + 5.0 * (-k * (T_euler_05 - T_inf))  # t = 10 min

print(f"Temperatur bei t = 10 min:")
print(f"  Analytisch:         {T_exakt_10:.4f} °C")
print(f"  solve_ivp (RK45):   {T_ivp_10:.4f} °C  "
      f"  Fehler: {abs(T_ivp_10 - T_exakt_10):.2e} °C")
print(f"  Euler (dt = 5 min): {T_euler_10:.4f} °C  "
      f"  Fehler: {abs(T_euler_10 - T_exakt_10):.4f} °C")

Das Ergebnis ist eindeutig: `solve_ivp` erreicht einen Fehler von unter
0.01 °C, während Euler mit $\Delta t = 5\,\text{min}$ mehr als 7 °C daneben
liegt. Wie viele Schritte `solve_ivp` dafür intern benötigt, sehen wir im
nächsten Unterabschnitt.

In [ ]:
# Euler mit dt = 5 min für den Vergleichsplot
T0_wert = 80.0; T_inf = 20.0; k = 0.1
n_e = 10; dt_e = 5.0
t_euler = np.linspace(0, 50, n_e + 1)
T_euler = np.zeros(n_e + 1)
T_euler[0] = T0_wert
for i in range(n_e):
    T_euler[i+1] = T_euler[i] + dt_e * (-k * (T_euler[i] - T_inf))

# Analytische Referenz
t_eval = sol.t
T_exakt = T_inf + (T0_wert - T_inf) * np.exp(-k * t_eval)

# Visualisierung
fig, ax = plt.subplots()
ax.plot(t_eval, T_exakt, label='analytisch (Referenz)')
ax.plot(t_eval, sol.y[0], linestyle=':', label='solve_ivp (RK45)')
ax.plot(t_euler, T_euler, marker='o', label='Euler ($\\Delta t = 5$ min)')
ax.set_xlabel('Zeit in min')
ax.set_ylabel('Temperatur in °C')
ax.set_title('Vergleich: Euler vs. solve_ivp vs. analytisch')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

Die Euler-Linie (durchgezogen mit Kreismarkern) weicht sichtbar von der
analytischen Referenz ab; die gepunktete `solve_ivp`-Linie liegt darauf.

**Hinweis**: `solve_ivp` erwartet immer die Signatur `f(t, y)`, wobei `t` an
erster Stelle steht und `y` ein Array ist. Für eine skalare DGL ist `y` ein
Array mit einem Element. Häufige Fehler sind:

* `f(y)` ohne `t`,
* `f(y, t)` in falscher Reihenfolge, oder
* `y` wie einen Skalar behandeln ohne `y[0]` zu schreiben.

### Mini-Übung 1

1. Was ist `sol.y.shape` in unserem Beispiel? Erklären Sie die beiden
   Dimensionen. Was würde `sol.y.shape` sein, wenn die DGL drei Zustandsgrößen
   hätte?
2. Was erwartet man für den Temperaturverlauf, wenn die Anfangstemperatur
   unter der Umgebungstemperatur liegt? Probieren Sie `y0=[5.0]` aus.
   Sagen Sie das Ergebnis zuerst voraus, bevor Sie den Code ausführen.
3. Warum gibt `f_abkuehlung` eine Liste `[dTdt]` zurück und keinen Skalar `dTdt`,
   obwohl die DGL nur eine Zustandsgröße hat?

In [ ]:
# Code-Zelle

## Adaptive Schrittweite: wie RK45 intern arbeitet

Das Euler-Verfahren benutzt eine feste, vom Anwender gewählte Schrittweite.
RK45 geht anders vor: Es schätzt bei jedem Schritt den lokalen Fehler ab
und verkleinert oder vergrößert die Schrittweite automatisch. Wo die
Lösung sich schnell ändert, werden kleine Schritte gewählt; wo sie fast
konstant ist, können große Schritte genügen.

In [ ]:
# --- solve_ivp OHNE t_eval: zeigt die tatsächlich verwendeten internen Schritte ---
sol_intern = solve_ivp(f_abkuehlung, t_span=(0, 50), y0=[T0_wert])

print(f"Anzahl interner Zeitpunkte (RK45):  {len(sol_intern.t)}")
print()
print("Schrittweiten (Differenz benachbarter interner Zeitpunkte):")
dt_intern = np.diff(sol_intern.t)
for i, (ti, dti) in enumerate(zip(sol_intern.t[:-1], dt_intern)):
    print(f"  Schritt {i+1:2d}: t = {ti:5.2f} --> {sol_intern.t[i+1]:5.2f} min  "
          f"(Δt = {dti:.4f} min)")

print()
print(f"Kleinste Schrittweite:  {dt_intern.min():.4f} min")
print(f"Größte Schrittweite:    {dt_intern.max():.4f} min")

Zusätzlich visualisieren wir die Schrittweiten.

In [ ]:
# --- Visualisierung der internen Schrittweiten ---
t_fein  = np.linspace(0, 50, 500)
T_fein  = T_inf + (T0_wert - T_inf) * np.exp(-k * t_fein)

fig, ax = plt.subplots()
ax.plot(t_fein, T_fein, label='analytisch')
ax.scatter(sol_intern.t, sol_intern.y[0], color='red',
                label=f'Interne RK45-Schritte ({len(sol_intern.t)} Punkte)')
ax.set_xlabel('Zeit in min')
ax.set_ylabel('Temperatur in °C')
ax.set_title('RK45: interne Auswertungspunkte')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

Die intern benötigten Schritte vergleichen wir mit der Anzahl an Schritten, die
das Euler-Verfahren für eine festgelegte Schrittweit braucht.

In [ ]:
# --- Effizienzvergleich: Anzahl Schritte vs. Fehler bei t = 10 min ---
T_exakt_10 = T_inf + (T0_wert - T_inf) * np.exp(-k * 10.0)

# Euler: verschiedene dt, jeweils Schritte bis t = 10 min
print(f"{'Methode                  ':}  {'Schritte   ':}  {'Fehler bei t=10 [°C]'}")
print("-" * 62)
for dt in [5.0, 2.0, 0.5, 0.1]:
    n_10 = int(10.0 / dt)   # Anzahl Schritte bis t = 10 min
    T  = T0_wert
    for _ in range(n_10):
        T = T + dt * (-k * (T - T_inf))
    print(f"Euler (dt = {dt:.1f} min)       {n_10:4d} \t\t {abs(T - T_exakt_10):.4f}")

# solve_ivp (ebenfalls bis t = 10 min)
sol_10 = solve_ivp(f_abkuehlung, t_span=(0, 10), y0=[T0_wert])
print()
print(f"{'solve_ivp (RK45)'}           {len(sol_10.t)}            "
      f"{abs(sol_10.y[0, -1] - T_exakt_10):.4f}")

Schon mit 4 internen Schritten liegt `solve_ivp` deutlich näher an der
analytischen Lösung als Euler selbst mit $\Delta t = 0{,}1$ min (100 Schritte
bis $t = 10$ min). Der Unterschied liegt im Algorithmus: Euler
approximiert die Ableitung durch einen einzigen Wert am linken Intervallrand.
RK45 wertet die rechte Seite $f(t, y)$ innerhalb jedes Schrittes an mehreren
Zwischenpunkten aus, kombiniert diese gewichtet und erzeugt so eine Näherung
höherer Ordnung (hier 4./5. Ordnung), mit deutlich kleinerem globalen Fehler
als beim Euler-Verfahren $O(\Delta t)$.

| Methode | Konvergenzordnung | Schrittwahl |
| --- | --- | --- |
| Euler | $O(\Delta t)$ | fest, manuell |
| RK45 in `solve_ivp` | $O(\Delta t^5)$ (Lösung) | adaptiv, automatisch |

Halbiert man die Schrittweite, sinkt der Fehler bei Euler ungefähr auf die
Hälfte, bei einem Verfahren 5. Ordnung etwa um den Faktor $2^5 =32$. RK45 liegt
in dieser Größenordnung; die adaptive Schrittweitensteuerung beeinflusst die
tatsächliche Schrittweite zusätzlich.

### Mini-Übung 2

1. Warum ist die erste Schrittweite von RK45 viel kleiner als die letzte?
   Begründen Sie in einem Satz, ohne Code auszuführen.
2. Würden Sie für $k = 1.0\,\text{min}^{-1}$ (zehnfach schnellere Abkühlung)
   mehr oder weniger interne RK45-Schritte erwarten? Testen Sie Ihre Vermutung.
3. Euler benötigt $\Delta t = 0.1\,\text{min}$ für einen Fehler von rund
   $0.1\,°C$ bei $t = 10\,\text{min}$ (vgl. Tabelle oben). Bei dieser
   Schrittweite braucht Euler für das gesamte Intervall $[0, 50]$
   500 Schritte. `solve_ivp` löst dasselbe Intervall mit 8 Schritten
   (vgl. `len(sol_intern.t)` weiter oben). Um welchen Faktor ist das
   effizienter?

In [ ]:
# Code-Zelle

## Nicht-autonome Differentialgleichungen

Bisher hing die rechte Seite $f(T)$ nur vom aktuellen Zustand ab, nicht explizit
von der Zeit. Solche Differentialgleichungen nennt man **autonom**. Wenn die
rechte Seite $f(t, T)$ auch die Zeit explizit enthält, spricht man von einer
**nicht-autonomen** DGL.

Ein konkretes Beispiel: Ein Metallstab liegt morgens draußen. Die
Außentemperatur steigt linear von $T_{\infty,0} = 10\,°C$ auf
$T_{\infty,0} + r \cdot t_\text{end} = 35\,°C$ innerhalb von 50 Minuten.
Der Stab hatte anfangs noch $T_0 = 80\,°C$ (vom Aufheizen über Nacht).

$$\dot{T}(t) = -k\bigl(T(t) - \underbrace{(T_{\infty,0} + r\,t)}_{T_\infty(t)}\bigr).$$

Die rechte Seite hängt nun explizit von $t$ ab: dasselbe $T$ führt bei
verschiedenen Zeiten zu verschiedenen Ableitungen.

In [ ]:
# --- Parameter ---
T0_wert  = 80.0   # Anfangstemperatur Metallstab in °C
T_inf0   = 10.0   # Außentemperatur bei t = 0 in °C
r_rampe  = 0.5    # Erwärmungsrate der Umgebung in °C/min
k        = 0.1    # Abkühlkonstante in 1/min
t_end    = 50.0

# --- Rechte Seite: nicht-autonom, da f explizit von t abhängt ---
# T_aussen(t) = T_inf0 + r_rampe * t  (linearer Anstieg)
def f_kuehl_rampe(t, y):
    T_aussen = T_inf0 + r_rampe * t       # Außentemperatur zum Zeitpunkt t
    dTdt = -k * (y[0] - T_aussen)         # Newtonsche Abkühlungsrate
    return [dTdt]

# --- solve_ivp löst nicht-autonome DGL genauso wie autonome ---
t_eval = np.linspace(0, t_end, 501)
sol_rampe = solve_ivp(fun=f_kuehl_rampe, t_span=(0, t_end),
                      y0=[T0_wert], t_eval=t_eval)

# --- Analytische Lösung als Probe ---
# T(t) = T_inf0 + r*t - r/k + (T0 - T_inf0 + r/k) * exp(-k*t)
# Die Lösung folgt der steigenden Außentemperatur mit konstantem Rückstand r/k.
T_ana = T_inf0 + r_rampe*t_eval - r_rampe/k + (T0_wert - T_inf0 + r_rampe/k) * np.exp(-k*t_eval)
T_aussen_verlauf = T_inf0 + r_rampe * t_eval   # zeitlicher Verlauf der Außentemperatur

print(f"Außentemperatur: {T_inf0:.1f} °C --> {T_aussen_verlauf[-1]:.1f} °C über {t_end:.0f} min")
print(f"Stabtemperatur bei t = {t_end:.0f} min: {sol_rampe.y[0, -1]:.2f} °C")
print(f"Analytisch:                    {T_ana[-1]:.2f} °C")
print(f"Rückstand (r/k = {r_rampe}/{k}):     "
      f"{T_aussen_verlauf[-1] - sol_rampe.y[0, -1]:.2f} °C  "
      f"(theoretisch: {r_rampe/k:.1f} °C)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_eval, T_aussen_verlauf, linestyle='--', label='Außentemperatur')
ax.plot(t_eval, T_ana,            label='analytisch (Probe)')
ax.plot(t_eval, sol_rampe.y[0],   linestyle=':', label='solve_ivp (RK45)')
ax.set_xlabel('Zeit in min')
ax.set_ylabel('Temperatur in °C')
ax.set_title('Nicht-autonome DGL: Stab folgt steigender Außentemperatur')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

Im eingeschwungenen Zustand folgt der Stab der Außentemperatur mit einem
konstanten Rückstand von $r/k = 0.5 / 0.1 = 5\,°C$. Das ist ein allgemeines
Ergebnis: Bei einer linear ansteigenden Außentemperatur liegt die
Stabtemperatur immer um $r/k$ dahinter. Mit einem kleinen $k$ (schlechte
Wärmeleitung) oder einem schnellen Anstieg $r$ vergrößert sich der Rückstand.

### Mini-Übung 3

1. Lesen Sie aus dem Plot ab: Wie groß ist der Rückstand des Stabs gegenüber
   der Außentemperatur am Ende (bei $t = 50$ min)? Stimmt er mit dem
   theoretischen Wert $r/k$ überein, und warum weicht er leicht ab?
2. Für eine autonome DGL reichte die Funktion `f_abkuehlung(T)` aus
   Abschnitt 10.1. Warum lässt sich `f_kuehl_rampe` nicht durch
   `f_abkuehlung` ersetzen, ohne den Code anzupassen?
3. Was würde passieren, wenn man die Aufwärmrate auf $r = 2.0\,°C/\text{min}$
   erhöht? Schätzen Sie den theoretischen Rückstand und überprüfen Sie mit
   `solve_ivp`.

In [ ]:
# Code-Zelle

## Zusammenfassung und Ausblick

`scipy.integrate.solve_ivp` löst gewöhnliche Differentialgleichungen numerisch
mit dem adaptiven RK45-Verfahren. Es braucht nur die Signatur `f(t, y)`, das
Integrationsintervall `t_span` und die Anfangsbedingung `y0` als Liste. Mit
wenigen internen Schritten erreicht RK45 für das Kühlproblem eine Genauigkeit,
die Euler mit 500 Schritten nicht schafft. Nicht-autonome DGLen, bei denen die
rechte Seite explizit von $t$ abhängt, werden ohne Anpassung des Solvers gelöst,
weil `f(t, y)` den Zeitpunkt immer als Argument erhält.

In Kapitel 11 werden wir Differentialgleichungen zweiter Ordnung (zum Beispiel
Schwingungsgleichungen) auf Systeme erster Ordnung zurückführen und mit
`solve_ivp` lösen. Dafür wird der Zustandsvektor `y` dann zwei Einträge haben:
Position und Geschwindigkeit.